# WQR — Wavelet Quantile Regression
## Comprehensive Tutorial Notebook

**Author:** Dr. Merwan Roudane  
**Email:** merwanroudane920@gmail.com  
**Package:** `wqr` v1.0.0  
**License:** MIT

---

This notebook demonstrates all **8 modules** of the `wqr` package:

| # | Module | Method | Reference |
|---|--------|--------|-----------|
| 1 | `qq_regression` | Quantile-on-Quantile Regression | Sim & Zhou (2015) |
| 2 | `wavelet_qr` | Wavelet Quantile Regression (WQR) | Adebayo & Ozkan (2023) |
| 3 | `multivariate_wqr` | Multivariate WQR (MWQR) | Adebayo et al. (2025) |
| 4 | `wavelet_qqr` | Wavelet QQR with P-values (WQQR) | Adebayo, Ozkan et al. (2025) |
| 5 | `causality` | Nonparametric Quantile Causality (WNQC) | Balcilar et al. (2016) |
| 6 | `mediation` | Wavelet Quantile Mediation & Moderation | Adebayo (2025) |
| 7 | `correlation` | Wavelet Quantile Correlation (WQC) | Roudane |
| 8 | `quantile_density` | Wavelet Quantile Density Estimation | Chesneau, Dewan & Doosti |

## 0. Setup & Data Generation

We generate a synthetic dataset that mimics a financial econometrics setting:
- **Y** = Dependent variable (e.g., stock returns)
- **X** = Independent variable (e.g., oil price changes)
- **Z** = Mediator/Moderator (e.g., policy uncertainty index)

In [ ]:
# Install if needed
# !pip install wqr

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import wqr
print(f"WQR version: {wqr.__version__}")

# ── Synthetic Data ──────────────────────────────────────────────
np.random.seed(2025)
n = 500
t = np.arange(n)

# Simulate with heterogeneous dependence structure
x = np.sin(2 * np.pi * t / 100) + np.random.normal(0, 1, n)
z = 0.4 * x + np.random.normal(0, 0.8, n)
# Y has tail-dependent relation to X
eps = np.random.standard_t(df=5, size=n) * 0.5
y = 0.5 * x + 0.3 * z + 0.2 * x * (np.abs(x) > 1) + eps

print(f"Sample size: {n}")
print(f"Y: mean={y.mean():.3f}, std={y.std():.3f}")
print(f"X: mean={x.mean():.3f}, std={x.std():.3f}")
print(f"Z: mean={z.mean():.3f}, std={z.std():.3f}")

In [ ]:
# Descriptive statistics
stats = wqr.summary_statistics({"Y": y, "X": x, "Z": z})
stats

---
## 1. Quantile-on-Quantile Regression (QQR)

The QQ regression (Sim & Zhou, 2015) estimates the dependence between quantiles of Y and quantiles of X.

For each x-quantile $\tau$, the method subsets data where $X \leq Q_X(\tau)$ and runs quantile regression of Y on X at each y-quantile $\theta$:

$$Q_{\theta}(Y | X \leq Q_X(\tau)) = \alpha(\theta, \tau) + \beta(\theta, \tau) \cdot X$$

This produces a **coefficient surface** $\beta(\theta, \tau)$ over a $\theta \times \tau$ grid.

In [ ]:
# Run QQ Regression
qq_result = wqr.qq_regression(
    y, x,
    y_quantiles=np.arange(0.05, 1.0, 0.05),
    x_quantiles=np.arange(0.05, 1.0, 0.05),
    verbose=True
)

qq_result.summary()

In [ ]:
# 3D Surface Plot (MATLAB-style)
fig, ax = wqr.plot_qq_3d(
    qq_result,
    colormap="jet",
    title="QQ Regression Coefficient Surface",
    elev=25, azim=-135,
    figsize=(12, 9)
)
plt.show()

In [ ]:
# Heatmap with significance stars
fig, ax = wqr.plot_qq_heatmap(
    qq_result,
    colormap="jet",
    title="QQ Regression Coefficient Heatmap",
    show_significance=True,
    figsize=(10, 8)
)
plt.show()

In [ ]:
# Contour Plot
fig, ax = wqr.plot_qq_contour(
    qq_result,
    colormap="jet",
    title="QQ Regression Contour Plot",
    levels=25,
    figsize=(10, 8)
)
plt.show()

In [ ]:
# Export coefficient matrix
coef_matrix = qq_result.to_matrix("coefficient")
print("Coefficient matrix shape:", coef_matrix.shape)
print("\nLaTeX table (first 40 lines):")
latex = qq_result.export_latex()
print(latex[:800])

---
## 2. Wavelet Quantile Regression (WQR)

WQR decomposes both the dependent and independent variables using the **Maximal Overlap Discrete Wavelet Transform (MODWT)**, then runs quantile regression at each wavelet scale.

Wavelet bands capture different frequency components:
- **Short-term** (D1+D2): High-frequency fluctuations
- **Medium-term** (D3+D4): Business-cycle dynamics  
- **Long-term** (D5+): Low-frequency structural trends

In [ ]:
# Run WQR
wqr_result = wqr.wavelet_qr(
    y, x,
    wavelet="la8",
    J=5,
    bands=True,
    verbose=True
)

wqr_result.summary()

In [ ]:
# WQR Band Heatmap with significance stars
fig, ax = wqr.plot_wqr_heatmap(
    wqr_result.coefficients,
    colormap="green_orange_red",
    title="Wavelet Quantile Regression",
    x_label="Quantiles",
    y_label="Time Horizons",
    figsize=(12, 5)
)
plt.show()

In [ ]:
# Results table with stars
tbl = wqr.results_table(wqr_result.coefficients)
tbl

In [ ]:
# LaTeX export
latex = wqr.export_latex(
    wqr_result.coefficients,
    caption="Wavelet Quantile Regression Results",
    label="tab:wqr"
)
print(latex)

---
## 3. Multivariate Wavelet Quantile Regression (MWQR)

Extension to **multiple independent variables**. Each variable is decomposed via MODWT, and quantile regression is performed jointly on all wavelet bands.

In [ ]:
# Run MWQR with Y ~ X + Z
mwqr_result = wqr.multivariate_wqr(
    y,
    {"Oil": x, "Policy": z},
    wavelet="la8",
    J=5,
    dep_name="Returns",
    verbose=True
)

mwqr_result.summary()

In [ ]:
# Heatmap for each variable
for var in mwqr_result.indep_names:
    sub = mwqr_result.get_variable(var)
    fig, ax = wqr.plot_wqr_heatmap(
        sub,
        beta_col="beta", pval_col="p_value",
        row_col="quantile", col_col="band",
        colormap="green_orange_red",
        title=f"MWQR: {var} Effect on Returns",
        figsize=(12, 4)
    )
    plt.show()

In [ ]:
# Significance matrix for Oil
sig = mwqr_result.significance_matrix("Oil")
sig

---
## 4. Wavelet Quantile-on-Quantile Regression (WQQR) with P-values

Combines wavelet decomposition with QQ regression using a **Gaussian kernel-weighted local polynomial quantile regression** (LPRQ). Produces both a coefficient surface and a p-value surface for formal inference.

In [ ]:
# Run WQQR on the long-term band
wqqr_result = wqr.wavelet_qqr(
    y, x,
    quantile_step=0.05,
    wavelet="la8",
    J=5,
    band="long",
    bandwidth=1.0,
    verbose=True
)

wqqr_result.summary()

In [ ]:
# WQQR 3D Surface
fig, ax = wqr.plot_wqqr_surface(
    wqqr_result,
    colormap="jet",
    title="WQQR Coefficient Surface",
    figsize=(12, 9)
)
plt.show()

In [ ]:
# P-value significance heatmap
fig, ax = wqr.plot_wqqr_pvalue_heatmap(
    wqqr_result,
    alpha=0.05,
    title="WQQR Significance Map (p < 0.05)",
    figsize=(8, 7)
)
plt.show()

In [ ]:
# WQR vs WQQR comparison
fig, ax = wqr.plot_wqr_vs_wqqr(
    wqqr_result,
    title="WQR vs WQQR Coefficient Comparison",
    figsize=(10, 5)
)
plt.show()

---
## 5. Nonparametric Quantile Causality (WNQC)

Tests whether $X_{t-1}$ Granger-causes $Y_t$ at quantile $\tau$ using the **Balcilar-Jeong-Nishiyama** nonparametric test (Song, Whang & Shin, 2012). Tests can target:
- **Causality in mean** (moment = 1)
- **Causality in variance** (moment = 2)

In [ ]:
# Test: Does X cause Y in mean?
caus_mean = wqr.np_quantile_causality(
    x, y,
    test_type="mean",
    q=np.arange(0.05, 1.0, 0.05)
)

caus_mean.summary()

In [ ]:
# Plot causality test
fig, ax = wqr.plot_causality(
    caus_mean,
    title="X -> Y: Causality in Mean",
    figsize=(10, 5)
)
plt.show()

In [ ]:
# Causality in variance
caus_var = wqr.np_quantile_causality(
    x, y,
    test_type="variance"
)

fig, ax = wqr.plot_causality(
    caus_var,
    title="X -> Y: Causality in Variance",
    figsize=(10, 5)
)
plt.show()

In [ ]:
# Wavelet-based causality (per frequency band)
wcaus = wqr.wavelet_np_causality(
    x, y,
    test_type="mean",
    wavelet="la8",
    J=5,
    bands=True,
    verbose=True
)

wcaus.summary()

In [ ]:
# Wavelet causality heatmap
fig, ax = wqr.plot_wavelet_causality(
    wcaus,
    colormap="green_orange_red",
    title="Wavelet NP Causality in Mean",
    figsize=(12, 5)
)
plt.show()

---
## 6. Wavelet Quantile Mediation & Moderation

Decomposes the causal pathway $X \to Z \to Y$ (mediation) and the interaction $X \times Z \to Y$ (moderation) at each wavelet scale and quantile.

Five components are estimated:
1. **Direct effect**: $Y \sim X$
2. **Interaction (moderation)**: $Y \sim X + Z + X \cdot Z$ (coefficient on $X \cdot Z$)
3. **Path a**: $Z \sim X$
4. **Path b**: $Y \sim X + Z$ (coefficient on $Z$)
5. **Indirect effect (mediation)**: $a \times b$

In [ ]:
# Run Mediation/Moderation
med_result = wqr.wavelet_mediation(
    y, x, z,
    wavelet="la8",
    dep_name="Returns",
    main_name="Oil",
    mod_name="Policy",
    verbose=True
)

med_result.summary()

In [ ]:
# Five-panel mediation plot
fig, axes = wqr.plot_mediation_panel(
    med_result,
    colormap="green_yellow_red",
    figsize=(14, 20)
)
plt.show()

---
## 7. Wavelet Quantile Correlation (WQC)

Computes the **quantile correlation** between wavelet-decomposed time series at each level, with **Monte Carlo confidence intervals** for statistical inference.

In [ ]:
# Run WQC
wqc_result = wqr.wavelet_quantile_correlation(
    x, y,
    quantiles=np.array([0.05, 0.25, 0.50, 0.75, 0.95]),
    wavelet="la8",
    J=5,
    n_sim=500,
    verbose=True
)

wqc_result.summary()

In [ ]:
# WQC Heatmap
fig, ax = wqr.plot_wqc_heatmap(
    wqc_result,
    colormap="viridis",
    title="Wavelet Quantile Correlation",
    figsize=(10, 6)
)
plt.show()

In [ ]:
# Quantile Correlation heatmap (non-wavelet)
fig, ax = wqr.plot_correlation_heatmap(
    y, x,
    x_label="X Quantiles",
    y_label="Y Quantiles",
    title="Quantile Correlation Matrix",
    figsize=(9, 8)
)
plt.show()

---
## 8. Wavelet Quantile Density Estimation

Nonparametric estimation of the **quantile density function** $q(p) = 1/f(F^{-1}(p))$ using wavelets.

Three estimators are compared:
1. **Linear wavelet estimate** (scaling function coefficients)
2. **Hard thresholding** (via DWT)
3. **Local linear smoothing** (Gaussian kernel)

We use a **Generalized Lambda Distribution (GLD)** sample where the true quantile density is known.

In [ ]:
# GLD sample (lambda1=0, lambda2=1, lambda3=0.5, lambda4=0.5)
np.random.seed(42)
n_sample = 500
u = np.random.uniform(0.001, 0.999, n_sample)
l1, l2, l3, l4 = 0, 1, 0.5, 0.5
gld_sample = l1 + (u**l3 - (1-u)**l4) / l2

qd_result = wqr.wavelet_quantile_density(
    gld_sample,
    j0=5,
    bandwidth=0.12,
    gld_params=(l1, l2, l3, l4)
)

qd_result.summary()

In [ ]:
# Four-panel quantile density plot
fig, axes = wqr.plot_quantile_density(
    qd_result,
    figsize=(12, 9)
)
plt.show()

---
## 9. Publication-Quality Tables

The `tables` module produces formatted tables with significance stars for direct inclusion in journal papers.

In [ ]:
# Formatted results table
tbl = wqr.results_table(wqr_result.coefficients)
tbl

In [ ]:
# LaTeX export
latex_code = wqr.export_latex(
    wqr_result.coefficients,
    caption="Wavelet Quantile Regression: Oil $\\rightarrow$ Stock Returns",
    label="tab:wqr_oil_stock"
)
print(latex_code)

In [ ]:
# Summary statistics
stats = wqr.summary_statistics({"Returns (Y)": y, "Oil (X)": x, "Policy (Z)": z})
stats

---
## 10. Saving Results

All result objects support CSV export and DataFrame conversion.

In [ ]:
# Export QQ regression results to CSV
qq_result.export_csv("qq_regression_results.csv")
print("Saved qq_regression_results.csv")

# Get DataFrames
qq_df = qq_result.to_dataframe()
wqr_df = wqr_result.coefficients
caus_df = caus_mean.to_dataframe()

print(f"\nQQ results: {len(qq_df)} rows")
print(f"WQR results: {len(wqr_df)} rows")
print(f"Causality results: {len(caus_df)} rows")

# Display causality results
caus_df

---
## References

1. Sim, N. & Zhou, H. (2015). *Oil prices, US stock return, and the dependence between their quantiles.* Journal of Banking & Finance, 55, 1-12.
2. Balcilar, M., Gupta, R. & Pierdzioch, C. (2016). *Does uncertainty move the gold price?* International Review of Financial Analysis.
3. Adebayo, T. S. & Ozkan, O. (2023). *Investigating the influence of socioeconomic conditions...* Journal of Cleaner Production.
4. Adebayo, T. S. et al. (2025). *Unpacking policy ambiguities...* Applied Economics.
5. Adebayo, T. S., Ozkan, O. et al. (2025). *Environmental Sciences Europe.*
6. Adebayo, T. S. (2025). *Can ESG Uncertainty Alter the Emissions Impact?* Statistical Journal of the IAOS.
7. Song, S., Whang, Y.-J. & Shin, Y. (2012). *Econometric Reviews.*
8. Chesneau, C., Dewan, I. & Doosti, H. *Nonparametric wavelet quantile density estimation.*

---

**End of tutorial.**  
For questions or issues, contact: merwanroudane920@gmail.com